In [156]:
import pandas as pd
from thefuzz import process
import numpy as np
data = pd.read_csv("C:\\Games\\Student Materials\\banking_finance_dirty_dataset.csv", dtype=str)
data.to_csv('data_backup_before_cleaning.csv', index=False)

print(data.head())
print(data.columns)
print(data.info())
print(data.describe().T)
print(data.isnull().sum()/len(data)*100)
print(data.duplicated().sum())

  customer_id transaction_id age  gender account_type       branch  \
0   CUST-2723      TXN802843  27    MALE      Savings  Los Angeles   
1   CUST-2134      TXN845682  39       M      SAVINGS  Los Angeles   
2   CUST-1534      TXN886476  43  FEMALE      Current         N.Y.   
3   CUST-1595      TXN608013  45  Female     checking         N.Y.   
4   CUST-1852      TXN132011  37     NaN           FD      Phoenix   

  account_open_date transaction_date balance_amount currency  \
0       17-Apr-2018       11/15/2018        1752.31      GBP   
1        04/04/2020       11/14/2022         989.62      usd   
2        22/03/2019       13/04/2020        7367.11      NGN   
3        06-30-2022       02/02/2024        1312.88      USD   
4          11/21/22       04/29/2024        5808.98      eur   

  transaction_amount transaction_type credit_score loan_amount loan_status  \
0             300.42          DEPOSIT          676     4527.56   Defaulted   
1             358.67              ATM 

ISSUES: NULL
1. age---------impute median
2. gender-------fill 'unknown'
3. branch--------impute mode
4. loan amount-----impute median
5. load_status------Fill 'unknown'
6. loan_approved-----Fill 'unknown'
8. annual_income-----impute median
9. email---------Fill 'Unknown'

In [157]:
print(data['account_type'].value_counts())
print(data['branch'].value_counts())
print(data['currency'].value_counts())
print(data['transaction_type'].value_counts())
print(data['loan_status'].value_counts())
print(data['loan_approved'].value_counts())
print(data['occupation'].value_counts())

account_type
FD               244
Savings          226
Checking         220
CHECKING         218
savings          212
Fixed Deposit    212
SAVINGS          211
Current          208
fixed deposit    196
checking         193
Name: count, dtype: int64
branch
Los Angeles    222
N.Y.           217
Phoenix        211
los angeles    209
Chicago        202
Houston        202
CHI            198
new york       197
New York       194
LA             178
Name: count, dtype: int64
currency
NGN     264
EUR     244
eur     241
 USD    241
usd     235
USD     234
GBP     232
gbp     225
USD     224
Name: count, dtype: int64
transaction_type
Deposit          258
Withdrawal       246
wire transfer    246
DEPOSIT          244
withdrawal       241
Transfer         237
ATM              234
Wire Transfer    218
atm              216
Name: count, dtype: int64
loan_status
Default      264
Closed       261
Defaulted    254
active       253
Active       247
ACTIVE       243
DEFAULT      236
closed       230
Name:

ISSUES:
1. account_type ------fix title, gaps
2. branch ------------fix title, CHI to Chicago, La to Los Angeles and NY to New York
3. currency ----------make every string CAPTITAL
4. transaction_type----fix title, make ATM capital, replace transfer to wire transfer
5. loan status --------fix title, repalce defaulted to default
6. loan approved -------fix title
7. occupation ---------fix title, replace enginr tp engineer, replace engpnr to engineer, replace Accountlnt to accountant and accountyant to accountant and Dzcter to doctor and toachr to tracher




In [ ]:
data = data.drop_duplicates()

data['account_type'] = data['account_type'].str.strip().str.title().replace('Fd', 'Fixed Deposit')

data['branch'] = data['branch'].str.strip().str.title()
data['branch'] = data['branch'].replace({
    'Chi': 'Chicago',
    'La': 'Los Angeles',
    'N.Y.': 'New York'
})

data['currency'] = data['currency'].str.strip().str.upper()

data['transaction_type'] = data['transaction_type'].str.strip().str.title().replace({
    'Atm': 'ATM',
    'Transfer': 'Wire Transfer'
})

data['loan_status'] = data['loan_status'].str.strip().str.title().replace({
    'Defaulted': 'Default'
})

data['loan_approved'] = data['loan_approved'].str.strip().str.title()




Negative values in numeric columns:
 Series([], dtype: float64)
Series([], dtype: float64)


Cleaning dirty strings to clean string using fuzzy matching

In [159]:
clean_occupations = ['Engineer', 'Doctor', 'Teacher', 'Lawyer', 'Accountant']

def match_occupation(value, choices, threshold=80):
    if pd.isnull(value):
        return 'unknown'
    result = process.extractOne(value, choices)
    if result[1] >= threshold:
        return result[0]
    else:
        return 'unknown'

data['occupation'] = data['occupation'].apply(
    lambda x: match_occupation(x, clean_occupations, threshold=80)
)

print(data['occupation'].value_counts(dropna=False))



occupation
Engineer      400
Accountant    399
Lawyer        392
Doctor        380
Teacher       345
unknown       184
Name: count, dtype: int64


In [ ]:
numeric_cols = data.select_dtypes(include=['number','float64', 'int64']).columns
negative_count = (data[numeric_cols] < 0).sum()
print("\nNegative Values Count:")
print(negative_count)

data['age'] = pd.to_numeric(data['age'], errors='coerce')
data.drop(data[data['age'] < 0].index, inplace=True)
data.drop(data[data['age'] > 100].index, inplace=True)
data['age'] = data['age'].fillna(data['age'].median())  

data['credit_score'].value_counts().head()
data['credit_score'] = pd.to_numeric(data['credit_score'], errors='coerce')
data.drop(data[data['credit_score'] < 350].index, inplace=True)
data.drop(data[data['credit_score'] > 850].index, inplace=True)
data['credit_score'] = data['credit_score'].fillna('Unknown')

data['gender'] = data['gender'].replace({'M': 'Male', 'F': 'Female'})
data['gender']=data['gender'].str.strip().str.title().fillna('Unknown')

data['account_open_date'] = pd.to_datetime(data['account_open_date'], format='mixed', errors='coerce')
data['transaction_date'] = pd.to_datetime(data['transaction_date'], format='mixed', errors='coerce')

data['branch'] = data['branch'].fillna(data['branch'].mode()[0])

data['balance_amount'] = pd.to_numeric(data['balance_amount'], errors='coerce')
data['balance_amount'] = data['balance_amount'].fillna(data['balance_amount'].median())

data['loan_amount'] = pd.to_numeric(data['loan_amount'], errors='coerce')
data['loan_amount'] = data['loan_amount'].fillna(data['loan_amount'].median())

data['loan_status'] = data['loan_status'].fillna('Unknown')

data['loan_approved'] = data['loan_approved'].fillna('Unknown')

data['occupation'] = data['occupation'].fillna('Unknown')

data['annual_income'] = pd.to_numeric(data['annual_income'], errors='coerce')
data['annual_income'] = data['annual_income'].fillna(data['annual_income'].median())


data['email'] = data['email'].fillna('Unknown')

data['notes'] = data['notes'].fillna('Unknown')

data['transaction_amount'] = pd.to_numeric(data['transaction_amount'], errors='coerce')
data['transaction_amount'] = data['transaction_amount'].fillna(data['transaction_amount'].median())

#flagged income values that are negative or very high
data['income_flag'] = data['annual_income'].apply(
    lambda x: 'Suspicious - Negative' if x < 0
    else 'Suspicious - Extreme' if x > 10000000
    else 'Normal'
)
print("\nIncome Flag Distribution:")
print(data['income_flag'].value_counts())


Negative Values Count:
age                     0
balance_amount         19
transaction_amount    219
loan_amount            68
annual_income         429
dtype: int64

Income Flag Distribution:
income_flag
Normal                   1531
Suspicious - Negative     429
Suspicious - Extreme       23
Name: count, dtype: int64


Changing str to datetime

In [164]:
print(data.head())
print(data.columns)
print(data.info())
print(data.describe().T)
print(data.isnull().sum()/len(data)*100)
print(data.duplicated().sum())

  customer_id transaction_id   age   gender   account_type       branch  \
0   CUST-2723      TXN802843  27.0     Male        Savings  Los Angeles   
1   CUST-2134      TXN845682  39.0     Male        Savings  Los Angeles   
2   CUST-1534      TXN886476  43.0   Female        Current     New York   
3   CUST-1595      TXN608013  45.0   Female       Checking     New York   
4   CUST-1852      TXN132011  37.0  Unknown  Fixed Deposit      Phoenix   

  account_open_date transaction_date  balance_amount currency  ...  \
0        2018-04-17       2018-11-15         1752.31      GBP  ...   
1        2020-04-04       2022-11-14          989.62      USD  ...   
2        2019-03-22       2020-04-13         7367.11      NGN  ...   
3        2022-06-30       2024-02-02         1312.88      USD  ...   
4        2022-11-21       2024-04-29         5808.98      EUR  ...   

   transaction_type credit_score loan_amount  loan_status loan_approved  \
0           Deposit        676.0     4527.56      Def